In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

In [3]:
import numpy as np
import pylab as plt

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy
from vimms_gym.evaluation import evaluate, evaluate_method

# 1. Parameters

In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (500, 2000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E5, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE
noise_density = 0.1
noise_max_val = 1e3

In [9]:
mzml_filename = 'Beer_multibeers_1_fullscan1.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)

2022-03-28 21:41:46.655 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans
2022-03-28 21:41:48.955 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans


In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
    },
    'noise': {
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [11]:
max_peaks = 100
out_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [12]:
total_timesteps = 20E6
n_eval_episodes = 10
deterministic = True

# 2. Training

## Train PPO

In [13]:
model_name = 'PPO'

In [14]:
# default parameters
learning_rate = 0.0003
batch_size = 64
n_steps = 2048
ent_coef = 0.0
gamma = 0.99
gae_lambda = 0.95

# modified parameters
# learning_rate = 0.0003
# batch_size = 64
# n_steps = 2048
# ent_coef = 0.01
# gamma = 0.80
# gae_lambda = 0.80

In [15]:
env = DDAEnv(max_peaks, params)
check_env(env)

### Multi-processing

In [16]:
from typing import Callable
import multiprocessing

def make_env(rank, seed=0):
    def _init():
        env = DDAEnv(max_peaks, params)
        env.seed(rank)
        return env
    set_random_seed(seed)
    return _init

num_cpu = multiprocessing.cpu_count()
num_cpu

192

In [17]:
if num_cpu > 100:
    num_cpu = 100

In [18]:
env = SubprocVecEnv([make_env(i) for i in range(num_cpu)]) 

In [19]:
tensorboard_log = './%s/%s_DDAEnv_tensorboard' % (out_dir, model_name)

model = PPO("MultiInputPolicy", env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
model.learn(total_timesteps=total_timesteps)

In [20]:
# model = PPO("MultiInputPolicy", env, verbose=2, 
#             learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
#             ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [21]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

 /home/joewandy/.conda/envs/vimms-gym/lib/python3.9/site-packages/stable_baselines3/common/evaluation.py:65: UserWarning:Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.


(627.9907808779133, 58.585167918988375)

In [22]:
fname = '%s/model_%s' % (out_dir, model_name)
model.save(fname)

# 2. Evaluation

Generate some chemical sets

In [23]:
n_eval_episodes = 1
out_dir = 'evaluation'
methods = [
    'random',
    'TopN',
    'PPO'    
]

In [24]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


Run different methods

In [25]:
max_peaks

100

In [26]:
results_dir = '%s/eval_%d_%d' % (out_dir, max_peaks, rt_tol)
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()

    env = DDAEnv(max_peaks, copy_params)
    evaluate_method(env, chem_list, method, results_dir, min_ms1_intensity=min_ms1_intensity, model=model)
    print()

method = random max_peaks = 100 rt_tol = 15

Episode 0 finished after 3962 timesteps with reward 540.3669175614414
(0.8524844720496895, 0.6821844472801313)

method = TopN max_peaks = 100 rt_tol = 15

Episode 0 finished after 3910 timesteps with reward 677.3758581024554
(0.9860248447204969, 0.9230554693020802)

method = PPO max_peaks = 100 rt_tol = 15

Episode 0 finished after 3071 timesteps with reward 609.6798043783126
(0.8618012422360248, 0.6490983168523725)



In [27]:
from mass_spec_utils.data_import.mzml import MZMLFile
import os

In [28]:
def count_ms1_ms2(mzml_file):
    mzml = MZMLFile(mzml_file)
    ms1 = len([sc for sc in mzml.scans if sc.ms_level == 1])
    ms2 = len([sc for sc in mzml.scans if sc.ms_level == 2])
    return ms1, ms2

In [29]:
results_dir

'evaluation/eval_100_15'

In [30]:
ppo_file = os.path.join(results_dir, 'PPO_0.mzML')
count_ms1_ms2(ppo_file)

(929, 2142)

In [31]:
random_file = os.path.join(results_dir, 'random_0.mzML')
count_ms1_ms2(random_file)

(38, 3924)

In [32]:
topN_file = os.path.join(results_dir, 'TopN_0.mzML')
count_ms1_ms2(topN_file)

(90, 3820)